<div>
  <b>Escuela Politécnica Nacional</b><br>
  <small>Facultad de Ingeniería de Sistemas</small><br>
  <small>Ingeniería en Ciencias de la Computación</small>
  <hr>
  <div style="display:flex; justify-content:space-between;">
    <div>
      Estudiante: <b>Mateo Cumbal</b><br>
      Fecha: <b>2026-06-30</b>
    </div>
    <div style="text-align:right;">
      Asignatura: <b>Recuperación de la Información</b><br>
      Paralelo: <b>GR1CC</b>
    </div>
  </div>
  <hr>
  <div style="text-align:center;">
    <h1><b>Ejercicio 9 — Uso de la API de Google Gemini</b></h1>
  </div>
</div>

En este ejercicio vamos a aprender a utilizar la API de Google Gemini, tanto en su forma básica (consulta directa) como combinada con un sistema de recuperación de información (retrieval) sobre el corpus 20 Newsgroups.

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica.

In [ ]:
# Ejecutar solo si el entorno no tiene las librerías instaladas
# !pip install scikit-learn sentence-transformers numpy pandas google-genai python-dotenv

In [2]:
import os
import numpy as np
import pandas as pd
from collections import deque
from dotenv import load_dotenv
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer, util
from google import genai

In [3]:
# La API key se lee desde el archivo .env (nunca se hardcodea en el notebook)
load_dotenv()
API_KEY = os.environ.get("GOOGLE_API_KEY")
client  = genai.Client(api_key=API_KEY)


def consultar_gemini(prompt):
    """Envía un prompt a Gemini 2.5 Flash, imprime la respuesta y la retorna."""
    respuesta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    print("\n--- Respuesta de Gemini ---")
    print(respuesta.text)
    print("---------------------------\n")
    return respuesta

In [ ]:
# Consulta directa, sin contexto adicional.
# El modelo responde bien gramaticalmente, pero puede alucinar cuando se le pregunta por datos específicos que no conoce: 
# en vez de admitir la falta de información,inventa una respuesta con total seguridad.
_ = consultar_gemini("¿En qué día nació Manuel Yánez?")


--- Respuesta de Gemini ---
Manuel Yáñez nació el **27 de enero de 1942**.
---------------------------



## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [5]:
# Corpus estándar de scikit-learn: ~18 000 publicaciones de foros de noticias
# (Usenet) agrupadas en 20 categorías temáticas (deportes, espacio, religión, etc.)
newsgroups = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"))

df_corpus = pd.DataFrame({
    "texto"     : newsgroups.data,
    "categoria" : [newsgroups.target_names[i] for i in newsgroups.target],
})

# Descartamos documentos vacíos o demasiado cortos (ruido típico de este corpus)
df_corpus["texto"] = df_corpus["texto"].str.strip()
df_corpus = df_corpus[df_corpus["texto"].str.len() > 50].reset_index(drop=True)

print(f"Documentos cargados: {len(df_corpus)}")
print(f"Categorías: {df_corpus['categoria'].nunique()}")
df_corpus.head()

Documentos cargados: 17886
Categorías: 20


,texto,categoria
0,I am sure some bashers of Pens fans are pretty...,rec.sport.hockey
1,My brother is in the market for a high-perform...,comp.sys.ibm.pc.hardware
2,Finally you said what you dream about. Mediter...,talk.politics.mideast
3,Think!\n\nIt's the SCSI card doing the DMA tra...,comp.sys.ibm.pc.hardware
4,1) I have an old Jasmine drive which I cann...,comp.sys.mac.hardware


### 2.2 Transformo a embeddings

In [ ]:
modelo = SentenceTransformer("all-MiniLM-L6-v2")

documentos = df_corpus["texto"].tolist()
corpus_embeddings = modelo.encode(documentos, convert_to_numpy=True)

print(f"Embeddings generados: {corpus_embeddings.shape}")

### 2.3 Creo una query y hago la búsqueda

In [7]:
def buscar_similitud(consulta, top_k=5):
    """Recupera los top_k documentos más relevantes para la consulta dada."""
    query_embedding = modelo.encode(consulta, convert_to_numpy=True)
    scores          = util.cos_sim(query_embedding, corpus_embeddings)[0].numpy()
    top_indices     = np.argsort(scores)[::-1][:top_k]

    resultados = []
    for rank, idx in enumerate(top_indices, start=1):
        resultados.append({
            "Ranking"   : rank,
            "Categoría" : df_corpus.loc[idx, "categoria"],
            "Fragmento" : df_corpus.loc[idx, "texto"][:300],
            "Similitud" : round(float(scores[idx]), 4),
        })

    return pd.DataFrame(resultados)

Obtengo los 5 documentos más similares a mi query

In [13]:
query   = "evidence of water or life on other planets"
results = buscar_similitud(query, top_k=5)
results

,Ranking,Categoría,Fragmento,Similitud
0,1,sci.space,This was all badly reported in the news. Ther...,0.4483
1,2,sci.space,Not quite true. One of the instruments on Mar...,0.4480
2,3,sci.space,"Some more references:\n\nS.H. Dole\n\n""Habitab...",0.4441
3,4,sci.space,What is the deal with life on Mars? I save th...,0.4265
4,5,sci.space,The European Space Agency has involvement with...,0.4024


---
## Valor agregado

## 3. RAG simulado

Se construye un prompt que combina la query del usuario con el contexto recuperado por el SRI, y se envía a Gemini para obtener una respuesta fundamentada en el corpus.

In [14]:
def construir_prompt_rag(consulta, contexto):
    """Construye el prompt RAG: combina la consulta del usuario con el contexto recuperado por el SRI."""
    return (
        f"El usuario busca: '{consulta}'\n\n"
        f"El sistema de recuperación encontró estos documentos relevantes:\n{contexto}\n\n"
        f"Dame la respuesta en base al contexto dado por el SRI"
    )

In [15]:
# Solo pasamos las columnas relevantes al modelo (sin el índice)
contexto = results[["Categoría", "Fragmento", "Similitud"]].to_string(index=False)
prompt   = construir_prompt_rag(query, contexto)

_ = consultar_gemini(prompt)


--- Respuesta de Gemini ---
Basándose en los documentos proporcionados:

*   **Evidencia de agua:** Los documentos no mencionan explícitamente ninguna evidencia de agua encontrada en otros planetas.
*   **Evidencia de vida:**
    *   El **Mars Observer** incluirá instrumentos dedicados a la búsqueda de posibles **sitios fósiles**, lo que indica una investigación activa de evidencia de vida pasada en Marte.
    *   Se discuten teorías sobre la **vida en Marte**, incluyendo referencias a "la cara" y otras hipótesis, aunque algunas de ellas son consideradas "débiles".
    *   También se mencionan conceptos como "planetas habitables" y la inferencia de que la vida podría formarse fácilmente.
---------------------------



## 4. Ventana de contexto

Se añade memoria conversacional usando un `deque` con `maxlen=3` (query anterior, respuesta anterior, query nueva), de modo que el modelo pueda resolver referencias implícitas a consultas previas.

In [16]:
context_window = deque(maxlen=3)

# --- Primera consulta (con RAG) ---
query1    = "evidence of water or life on other planets"
results1  = buscar_similitud(query1, top_k=5)
contexto1 = results1[["Categoría", "Fragmento", "Similitud"]].to_string(index=False)
prompt1   = construir_prompt_rag(query1, contexto1)

response1 = consultar_gemini(prompt1)

# Guardamos la query y la respuesta para que la próxima consulta tenga contexto
context_window.append(query1)
context_window.append(response1.text)


--- Respuesta de Gemini ---
Basándose en el contexto proporcionado:

Los documentos sugieren que ha habido una búsqueda y discusión activa sobre la vida en otros planetas, especialmente en Marte:

*   Se menciona que uno de los instrumentos del *Mars Observer* estará buscando **potenciales sitios fósiles** en Marte, lo que indica un esfuerzo por encontrar evidencia de vida pasada (Documento 2).
*   Existen teorías y discusiones sobre la **vida en Marte**, incluyendo la referencia a "la 'cara'", aunque estas teorías son recibidas con cierto escepticismo en uno de los documentos (Documento 4).
*   Se hace referencia a argumentos sobre la facilidad con la que la vida podría surgir, basada en la hipótesis de que si la vida apareció poco después de un evento esterilizador, entonces podría formarse fácilmente (Documento 1).
*   También se mencionan estudios y publicaciones sobre "Planetas Habitables para el Hombre" y "Sistemas Planetarios Extra-Solares", lo que apunta a la investigación sob

In [17]:
# --- Segunda consulta (ambigua — depende del contexto anterior) ---
query2 = "y de esos documentos, cual parece mas creible o cientifico?"
context_window.append(query2)

# Gemini recibe el historial completo: query1 + respuesta1 + query2
_ = consultar_gemini("\n".join(context_window))


--- Respuesta de Gemini ---
Basándose en los documentos proporcionados y el análisis previo, el documento que parece más **creíble y científico** es el **Documento 2**.

**Razón:**

*   **Documento 2:** Menciona directamente un instrumento del **Mars Observer** buscando "potenciales sitios fósiles en Marte". Esto es una referencia a una **misión espacial real** (Mars Observer fue una misión de la NASA), un **instrumento específico** y un **objetivo científico claro y medible** (buscar evidencia de vida pasada a través de fósiles). Se basa en la exploración directa y la recolección de datos, que son pilares del método científico.

Aunque los otros documentos también tienen elementos científicos:

*   **Documento 3** ("estudios y publicaciones sobre 'Planetas Habitables para el Hombre' y 'Sistemas Planetarios Extra-Solares'") también es muy científico, ya que se refiere a campos de investigación y publicaciones. Es un fuerte candidato, pero menos específico sobre una misión o instrument